# LFW Grad-CAM — 05. Population saliency–compression join

전체 원본 공간 특징과 압축 민감도를
`extraction_uid + dataset_id + sample_id + model_uid` 및 원본 embedding
lineage로 엄격히 결합합니다. 모델과 압축 profile을 pooling하지 않고
identity-cluster bootstrap 상관을 계산합니다.


In [1]:
# cell 1 : 실행 코드
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_PROFILE = "arcface_ms1mv3_r100"     # arcface, adaface, magface 중 이번 실행 profile
MODE = "dev"               # 빠른 검증은 dev, 전체 논문 실행만 real
DATA_FRACTION = 1       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합과 random control의 재현 seed
EXECUTE_STAGE = True      # 필수 입력을 채우고 이 단계 계산 시에만 True
WRITE_OUTPUTS = True      # 새 immutable artifact 저장 시에만 True

all_profiles = CONFIG["models"]["selected_profiles"] + CONFIG["models"].get("bridge_profiles", [])
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    raise RuntimeError(f"차단된 profile입니다: {MODEL_PROFILE}")
if MODEL_PROFILE not in available_profiles:
    raise ValueError(f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}")
PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [2]:
# cell 2 : 실행 코드
import pandas as pd

from research.evaluation import (
    join_population_saliency_with_compression,
    saliency_compression_associations,
)
from research.explainability.gradcam import (
    read_population_saliency_features,
)

SALIENCY_ARTIFACT_DIR = None
LINEAGED_PAIRED_METRICS_PATH = None
LINEAGED_RETRIEVAL_METRICS_PATH = None
JOINED_OUTPUT_PATH = None
ASSOCIATION_OUTPUT_PATH = None
BOOTSTRAP_REPEATS = 500

# 자동 기본값 및 경로 탐색
if SALIENCY_ARTIFACT_DIR is None or LINEAGED_PAIRED_METRICS_PATH is None or LINEAGED_RETRIEVAL_METRICS_PATH is None:
    lfw_runs_root = PROJECT_ROOT / CONFIG["run"]["root"] / "lfw"
    latest_saliency_manifests = sorted(lfw_runs_root.rglob("saliency_population/manifest.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if latest_saliency_manifests:
        sal_dir = latest_saliency_manifests[0].parent
        run_dir = sal_dir.parent
        if SALIENCY_ARTIFACT_DIR is None:
            SALIENCY_ARTIFACT_DIR = sal_dir
        if LINEAGED_PAIRED_METRICS_PATH is None and (run_dir / "lineaged_paired.parquet").is_file():
            LINEAGED_PAIRED_METRICS_PATH = run_dir / "lineaged_paired.parquet"
        if LINEAGED_RETRIEVAL_METRICS_PATH is None and (run_dir / "lineaged_retrieval.parquet").is_file():
            LINEAGED_RETRIEVAL_METRICS_PATH = run_dir / "lineaged_retrieval.parquet"
        if JOINED_OUTPUT_PATH is None:
            JOINED_OUTPUT_PATH = run_dir / "joined_saliency_compression.parquet"
        if ASSOCIATION_OUTPUT_PATH is None:
            ASSOCIATION_OUTPUT_PATH = run_dir / "saliency_compression_associations.parquet"


In [3]:
# cell 3 : 실행 코드
if EXECUTE_STAGE:
    required = {
        "saliency": SALIENCY_ARTIFACT_DIR,
        "paired": LINEAGED_PAIRED_METRICS_PATH,
        "retrieval": LINEAGED_RETRIEVAL_METRICS_PATH,
    }
    missing = [name for name, value in required.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    saliency = read_population_saliency_features(required["saliency"])
    distortion = pd.read_parquet(required["paired"])
    retrieval = pd.read_parquet(required["retrieval"])
    joined = join_population_saliency_with_compression(
        saliency,
        distortion,
        retrieval_sensitivity=retrieval,
    )
    analysis_rows = joined.loc[
        joined["saliency_target_eligible"].astype(bool)
        & joined["heatmap_available"].astype(bool)
        & joined["gradcam_valid_heatmap"].fillna(False).astype(bool)
    ].copy()
    associations = saliency_compression_associations(
        analysis_rows,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        seed=SEED,
    )
    join_summary = {
        "joined_rows": int(len(joined)),
        "analysis_rows": int(len(analysis_rows)),
        "association_rows": int(len(associations)),
        "models": sorted(joined["model_uid"].astype(str).unique()),
        "profiles": sorted(
            joined["compression_profile"].astype(str).unique()
        ),
    }
    if WRITE_OUTPUTS:
        if JOINED_OUTPUT_PATH is None or ASSOCIATION_OUTPUT_PATH is None:
            raise RuntimeError("결합·상관 출력 경로를 모두 지정하세요.")
        for destination_value, frame in (
            (JOINED_OUTPUT_PATH, joined),
            (ASSOCIATION_OUTPUT_PATH, associations),
        ):
            destination = Path(destination_value).resolve()
            if destination.exists():
                raise FileExistsError(
                    f"기존 결과를 덮어쓸 수 없습니다: {destination}"
                )
            destination.parent.mkdir(parents=True, exist_ok=True)
            frame.to_parquet(destination, index=False)
else:
    join_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
join_summary


{'joined_rows': 79170,
 'analysis_rows': 54732,
 'association_rows': 540,
 'models': ['arcface-7972a704552df378345f'],
 'profiles': ['origin_512',
  'pca_128',
  'pca_256',
  'pca_32',
  'pca_384',
  'pca_64']}

동일 이미지가 여러 profile에 반복되므로 전체 행을 한 번에 pooling한
상관계수는 논문 결과로 사용하지 않습니다.
